# Coembedding

`rx.coembed` generates matched reactant and product conformers in one coordinate frame. Each endpoint keeps its own bonds and specified stereo. Pairs are sorted by combined score. Use `pair[0]` to select the lowest-score pair.

Both endpoints need the same atoms, including hydrogens and isotopes. Atom maps establish correspondence independently of atom order. Keep transferred hydrogens explicit and mapped.

The five examples use the public API and inline (CX)SMILES. Run cells in order. GIF overlays require `xyzrender`, which is included in the development environment (`uv sync`). The final section describes the supported stereo and its limits.


## How coembedding works

The following steps describe the default workflow and the options for generating initial coordinates. These options do not add stereo guarantees.

1. Match atoms and identify spectators and reacting atoms from connectivity, bond-order and declared stereo changes. `react_atoms` can add reacting pairs.
2. With `seed_union=True`, build a numerical union graph with RDKit ZERO bonds for changing edges. Average overlapping endpoint bounds. Use their interval hull when they are disjoint. This temporary graph does not replace either endpoint's chemistry.
3. With `seed_cpci=True`, add RDKit Custom Pairwise Coulombic Interactions (CPCI). These attract atoms on changing edges and repel unbonded outer atoms of a forming/breaking edge pair that shares a center. The coefficients are numerical preferences, not physical partial charges. `seed_cpci=False` removes these generated preferences without changing the union graph or endpoint bounds.
4. Copy the shared coordinates onto both endpoint graphs and prepare their declared stereo. Each independent UFF preparation attempt gets at most `seed_relax_iters=100` steps. Zero skips this minimization. Rejected preparation restores the shared coordinates.
5. Joint minimization alternates between endpoints. Each endpoint gets up to `joint_max_iters=100` native steps per cycle, for at most `max_cycles=100` cycles. All atoms can move. The objective is

$$E_R+E_P+\frac{w_S}{|S|}\sum_{i\in S}\|r_i-p_i\|^2+\frac{w_R}{|R|}\sum_{i\in R}\|r_i-p_i\|^2.$$

`alignment_weight=25` weights spectators. `react_weight=1.25` weights reacting atoms. Increasing either weight penalizes separation between the corresponding atoms more strongly. Increase `react_weight` when alignment at the reaction center matters more. Empty selections contribute zero. If joint minimization fails, a valid prepared pair can be retained.

`seed_union=False` instead embeds the endpoint with more bonds and transfers its coordinates. Ties favor the endpoint with fewer haptic sites. This option is independent of `seed_cpci`. Caller-supplied `EmbedParams.native` retains its own CPCI settings. The switch neither overwrites nor disables those settings.

A fixed `seed` reproduces the search and its retry sequence. Missing or rejected union candidates are replaced using fresh endpoint coordinates without generated CPCI. Successful pairs are retained. `joint_results` records the actual `seed_bounds`, `seed_cpci` and optimization outcome for each returned pair. `optimizer_attempts` also includes rejected joint attempts. Native preparation step counts are unavailable.


In [ ]:
import rxembed as rx
import xyzrender
from IPython.display import display


## 1-bromopentane and chloride

Chloride replaces bromide in `CCCCCBr.[Cl-]`. The mapped endpoints keep the carbon chain and both halides in correspondence. This quick start requests ten pairs and enables verbose diagnostics.


In [ ]:
pentyl_reactant_smiles = '[CH3:1][CH2:2][CH2:3][CH2:4][CH2:5][Br:6].[Cl-:7]'
pentyl_product_smiles = '[CH3:1][CH2:2][CH2:3][CH2:4][CH2:5][Cl:7].[Br-:6]'

pentyl_pair = rx.coembed(
    pentyl_reactant_smiles,
    pentyl_product_smiles,
    n=10,
    seed=42,
    threads=1,
    verbose=True,
    react_weight=10,
)
print(f"Accepted pairs: {len(pentyl_pair)}")
pentyl_reactant, pentyl_product = pentyl_pair[0]
display(xyzrender.render_gif(
    xyzrender.load(pentyl_reactant),
    overlay=xyzrender.load(pentyl_product),
    auto_align=False,
    gif_rot=True,
    rot_frames=36,
    gif_fps=8,
    hy=True,
))


## SN2

Fluoride replaces chloride in methyl chloride. Explicit mapped hydrogens keep the atom correspondence consistent between endpoints.


In [ ]:
sn2_reactant_smiles = '[C:1]([H:2])([H:3])([H:4])[Cl:6].[F-:5]'
sn2_product_smiles = '[C:1]([H:2])([H:3])([H:4])[F:5].[Cl-:6]'

sn2_pair = rx.coembed(
    sn2_reactant_smiles,
    sn2_product_smiles,
    n=10,
    seed=42,
    threads=1,
    verbose=True,
    charge=-1,
)
print(f"Accepted pairs: {len(sn2_pair)}")
sn2_reactant, sn2_product = sn2_pair[0]
display(xyzrender.render_gif(
    xyzrender.load(sn2_reactant),
    overlay=xyzrender.load(sn2_product),
    auto_align=False,
    gif_rot=True,
    rot_frames=36,
    gif_fps=8,
    hy=True,
))


## Ru carbonyl rearrangement (`ru-co`)

The Ru-bound carbamoyl becomes an O-bound formamide as a proton transfers from the pincer nitrogen. The CXSMILES donor-slot annotations preserve the selected octahedral coordination identity.


In [ ]:
ru_co_reactant_smiles = '[Ru+2:1]12(<-[P:2]([c:6]3[c:10]([H:11])[c:41]([H:42])[c:49]([H:50])[c:34]([H:35])[c:36]3[H:37])([c:13]3[c:16]([H:17])[c:53]([H:54])[c:59]([H:60])[c:57]([H:58])[c:39]3[H:40])[C:23]([H:24])([H:25])[C:27]([N:4]->1([C:7]([H:8])([H:9])[C:20]([P:3]->2([c:12]1[c:18]([H:19])[c:30]([H:31])[c:51]([H:52])[c:45]([H:46])[c:32]1[H:33])[c:26]1[c:14]([H:15])[c:43]([H:44])[c:55]([H:56])[c:61]([H:62])[c:47]1[H:48])([H:21])[H:22])[H:67])([H:28])[H:29])(<-[C-:38]#[O+:5])(<-[H-:63])<-[C-:65](=[O:64])[N:66]([H:68])[C:69]([H:70])([H:71])[H:72] |atomProp:0.atomNote.OCT:1.atomNote.s0:28.atomNote.s2:33.atomNote.s1:61.atomNote.s3:63.atomNote.s4:64.atomNote.s5|'
ru_co_product_smiles = '[Ru+2:1]12(<-[P:2]([c:6]3[c:10]([H:11])[c:41]([H:42])[c:49]([H:50])[c:34]([H:35])[c:36]3[H:37])([c:13]3[c:16]([H:17])[c:53]([H:54])[c:59]([H:60])[c:57]([H:58])[c:39]3[H:40])[C:23]([H:24])([H:25])[C:27]([N-:4]->1[C:7]([H:8])([H:9])[C:20]([P:3]->2([c:12]1[c:18]([H:19])[c:30]([H:31])[c:51]([H:52])[c:45]([H:46])[c:32]1[H:33])[c:26]1[c:14]([H:15])[c:43]([H:44])[c:55]([H:56])[c:61]([H:62])[c:47]1[H:48])([H:21])[H:22])([H:28])[H:29])(<-[C-:38]#[O+:5])(<-[H-:63])<-[O:64]=[C:65]([N:66]([H:68])[C:69]([H:70])([H:71])[H:72])[H:67] |atomProp:0.atomNote.OCT:1.atomNote.s0:28.atomNote.s2:33.atomNote.s1:60.atomNote.s3:62.atomNote.s4:63.atomNote.s5|'

ru_co_pair = rx.coembed(
    ru_co_reactant_smiles,
    ru_co_product_smiles,
    n=5,
    seed=42,
    verbose=True,
    charge=0,
)
print(f"Accepted pairs: {len(ru_co_pair)}")
ru_co_reactant, ru_co_product = ru_co_pair[0]
display(xyzrender.render_gif(
    xyzrender.load(ru_co_reactant),
    overlay=xyzrender.load(ru_co_product),
    auto_align=False,
    gif_rot=True,
    rot_frames=36,
    gif_fps=8,
    hy=True,
))


## Spiro fragmentation (`spiro-ts1`)

The C–N bond to morpholine breaks, and the oxyanion becomes a carbonyl. The remaining declared spiro-framework stereochemistry is retained. The example name refers to an endpoint pair, not an optimized transition state.


In [ ]:
spiro_ts1_reactant_smiles = '[O:1]=[C:2]1[N:3]([c:4]2[c:5]([H:31])[c:6]([H:32])[c:7]([H:33])[c:8]([H:34])[c:9]2[H:35])[N:10]=[C:11]([C:12]([H:36])([H:37])[H:38])[C@@:13]12[O:14][C@:15]([N+:16]1([H:17])[C:18]([H:39])([H:40])[C:19]([H:41])([H:42])[O:20][C:21]([H:43])([H:44])[C:22]1([H:45])[H:46])([O-:23])[C@@:24]2([c:25]1[c:26]([H:48])[c:27]([H:49])[c:28]([H:50])[c:29]([H:51])[c:30]1[H:52])[H:47]'
spiro_ts1_product_smiles = '[N:16]1([H:17])[C:18]([H:39])([H:40])[C:19]([H:41])([H:42])[O:20][C:21]([H:43])([H:44])[C:22]1([H:45])[H:46].[O:1]=[C:2]1[N:3]([c:4]2[c:5]([H:31])[c:6]([H:32])[c:7]([H:33])[c:8]([H:34])[c:9]2[H:35])[N:10]=[C:11]([C:12]([H:36])([H:37])[H:38])[C@@:13]12[O:14][C:15](=[O:23])[C@@:24]2([c:25]1[c:26]([H:48])[c:27]([H:49])[c:28]([H:50])[c:29]([H:51])[c:30]1[H:52])[H:47]'

spiro_ts1_pair = rx.coembed(
    spiro_ts1_reactant_smiles,
    spiro_ts1_product_smiles,
    n=5,
    seed=42,
    threads=1,
    verbose=True,
    charge=0,
)
print(f"Accepted pairs: {len(spiro_ts1_pair)}")
spiro_ts1_reactant, spiro_ts1_product = spiro_ts1_pair[0]
display(xyzrender.render_gif(
    xyzrender.load(spiro_ts1_reactant),
    overlay=xyzrender.load(spiro_ts1_product),
    auto_align=False,
    gif_rot=True,
    rot_frames=36,
    gif_fps=8,
    hy=True,
))


## Diels–Alder cycloaddition

Butadiene and methyl acrylate form two C–C bonds and a cyclohexene ring. The mapped endpoints retain the same atoms as connectivity and bond orders change. The new product stereocenter is unspecified, so `stereo="free"` permits either configuration.


In [ ]:
diels_alder_reactant_smiles = '[C:1](=[C:2]([C:3](=[C:4]([H:15])[H:16])[H:14])[H:13])([H:11])[H:12].[C:5](=[C:6]([C:7](=[O:8])[O:9][C:10]([H:20])([H:21])[H:22])[H:19])([H:17])[H:18]'
diels_alder_product_smiles = '[C:1]1([H:11])([H:12])[C:2]([H:13])=[C:3]([H:14])[C:4]([H:15])([H:16])[C:5]([H:17])([H:18])[C:6]1([C:7](=[O:8])[O:9][C:10]([H:20])([H:21])[H:22])[H:19]'

diels_alder_pair = rx.coembed(
    diels_alder_reactant_smiles,
    diels_alder_product_smiles,
    n=10,
    seed=42,
    threads=1,
    verbose=True,
    charge=0,
    stereo="free",
)
print(f"Accepted pairs: {len(diels_alder_pair)}")
diels_alder_reactant, diels_alder_product = diels_alder_pair[0]
display(xyzrender.render_gif(
    xyzrender.load(diels_alder_reactant),
    overlay=xyzrender.load(diels_alder_product),
    auto_align=False,
    gif_rot=True,
    rot_frames=36,
    gif_fps=8,
    hy=True,
))


## Which stereo is supported?

Specify one identity per endpoint. The product configuration can differ from the reactant configuration. Each configuration belongs to its endpoint graph. Atom maps establish correspondence between the graphs.

`@`/`@@` and `wU`/`wD` depend on local atom and bond order. Comparing these symbols alone does not establish retention or inversion. R/S labels can also change when ligand priorities change without physical inversion.

| Element | Supported input and behavior | Limits |
|---|---|---|
| Tetrahedral point stereo | Explicit `@`/`@@` centers can retain or change configuration between endpoints. Each returned endpoint is checked against its own declared configuration. | A stated target does not guarantee successful embedding. Inseparable ring branches or metal-containing branches can prevent local inversion. Recovery may also fail. |
| Pyramidal ligand/donor stereo | Graph-supported pyramidal N/P/As centers and selected coordinated donor hands are checked. Tests cover inversion at a bound amine. | Ordinary amines are not all treated as configurationally fixed. Metal `stereo="free"` selection can release labile or unassigned donor hands. |
| Double-bond E/Z | Slash bonds or RDKit E/Z tags specify each endpoint's independent E/Z configuration. The endpoints can have different configurations. | A bond that becomes single has no E/Z target. E/Z fixed by metal coordination belongs to the selected coordination identity rather than an independent ligand choice. |
| Native atropisomer axes | RDKit `STEREOATROPCW`/`STEREOATROPCCW` tags or CXSMILES such as `wU`/`wD` specify each endpoint's axial hand. The hands can differ. | An unlabelled torsion is not an axial-stereo constraint. General allene/cumulene axial stereo is outside this contract. A changing cyclic axis requires explicit spectator selection. Setting `spectator_atoms`/`react_atoms` does not guarantee successful embedding. |
| Metal coordination identity | Pass one selected `rx.metal(...)[index]` per endpoint or its `rx.cxsmiles(...)`. The selected geometry, donor slots, represented metal hand and haptic face/winding are validated. Endpoint identities can differ. | A geometry such as `metal="OCT"` can leave multiple arrangements or ligand configurations. Coembedding raises rather than choosing one. Specify ligand stereo too. |
| Planar/helical chirality from geometry | A coordinate reference can supply a handedness filter through the `workflow` extra. | The filter compares distinct perceived hands rather than exact sites or counts. Undetected elements can escape it. A mixture containing both hands cannot identify which site inverted. The filter does not construct general planar/helical inversions. A missing backend or caught reference-perception failure warns and disables that filter. Other adaptation errors can still raise. |

### Defaults and `stereo="free"`

- Coordinate-free organic endpoints default to `stereo="unassigned"`. Coembedding raises if undefined stereo expands into several identities. Assign each intended endpoint explicitly. Coembedding does not enumerate every endpoint pairing.
- Coordinate inputs default to `stereo="all"`. This adds comparison with the endpoint's input geometry when a reference can be built. Planar/helical comparison requires `rxembed[workflow]`. Native declared point, E/Z and atrop checks do not.
- For organic inputs, `stereo="free"` allows unspecified centers and disables the geometry-reference filter. Explicit point, E/Z and atrop tags remain enforced. Remove an input tag explicitly to remove its target.
- For metal inputs, `stereo="free"` can release labile donor point configurations during adaptation. Use fully selected isomers with the default mode when donor configuration matters. Selected coordination identity remains enforced in either mode.

The examples illustrate distinct contracts. Diels–Alder leaves the new product center unspecified and uses `stereo="free"`. The spiro example states its remaining carbon configurations. The Ru example encodes its selected coordination identity in CXSMILES donor-slot annotations.

Stereo restraints guide minimization. Final checks decide acceptance. Rejected pairs are replaced or removed, so fewer pairs may survive than requested. The call raises if none survives. Passing these checks does not establish a transition state or a favorable reaction approach.

`verbose=True` reports optimization diagnostics. Request more than three conformers (`n<=3` warns). Preserve the shared frame with `auto_align=False` in GIFs and `align=False` in XYZ exports.
